<a href="https://colab.research.google.com/github/gopika-vit/Projects-AI/blob/projects-in-colab/generator_degenerator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import os

# ─── CONFIG ───────────────────────────
LATENT_DIM = 128
BATCH_SIZE = 64
EPOCHS = 10000
CRITIC_STEPS = 5
LAMBDA_GP = 10
SAVE_DIR = "wgan_output"
os.makedirs(SAVE_DIR, exist_ok=True)

# ─── LOAD DATA ───────────────────────
def load_cifar():
    (x_train, _), _ = keras.datasets.cifar10.load_data()
    return (x_train.astype("float32") / 127.5) - 1.0

# ─── GENERATOR ───────────────────────
def build_generator():
    return keras.Sequential([
        layers.Dense(4*4*512, input_dim=LATENT_DIM),
        layers.Reshape((4,4,512)),

        layers.Conv2DTranspose(256, 4, 2, "same"),
        layers.BatchNormalization(),
        layers.ReLU(),

        layers.Conv2DTranspose(128, 4, 2, "same"),
        layers.BatchNormalization(),
        layers.ReLU(),

        layers.Conv2DTranspose(64, 4, 2, "same"),
        layers.BatchNormalization(),
        layers.ReLU(),

        layers.Conv2D(3, 3, padding="same", activation="tanh")
    ])

# ─── CRITIC (no sigmoid) ─────────────
def build_critic():
    return keras.Sequential([
        layers.Conv2D(64, 4, 2, "same", input_shape=(32,32,3)),
        layers.LeakyReLU(0.2),

        layers.Conv2D(128, 4, 2, "same"),
        layers.LeakyReLU(0.2),

        layers.Conv2D(256, 4, 2, "same"),
        layers.LeakyReLU(0.2),

        layers.Flatten(),
        layers.Dense(1)
    ])

# ─── GRADIENT PENALTY ────────────────
def gradient_penalty(critic, real, fake):
    alpha = tf.random.uniform([BATCH_SIZE,1,1,1], 0., 1.)
    interpolated = alpha * real + (1 - alpha) * fake

    with tf.GradientTape() as tape:
        tape.watch(interpolated)
        pred = critic(interpolated)

    grads = tape.gradient(pred, interpolated)
    norm = tf.sqrt(tf.reduce_sum(tf.square(grads), axis=[1,2,3]))
    return tf.reduce_mean((norm - 1.0)**2)

# ─── SAVE IMAGES ─────────────────────
def save_images(gen, step):
    noise = np.random.normal(0,1,(25,LATENT_DIM))
    imgs = gen.predict(noise, verbose=0)
    imgs = 0.5 * imgs + 0.5

    fig, axs = plt.subplots(5,5, figsize=(5,5))
    idx = 0
    for i in range(5):
        for j in range(5):
            axs[i,j].imshow(imgs[idx])
            axs[i,j].axis("off")
            idx += 1
    plt.savefig(f"{SAVE_DIR}/step_{step}.png")
    plt.close()

# ─── TRAIN ───────────────────────────
def train():
    data = load_cifar()
    gen = build_generator()
    critic = build_critic()

    g_opt = keras.optimizers.Adam(0.0001, beta_1=0.5)
    c_opt = keras.optimizers.Adam(0.0001, beta_1=0.5)

    for step in range(EPOCHS):

        # ── Train Critic ──
        for _ in range(CRITIC_STEPS):
            idx = np.random.randint(0, data.shape[0], BATCH_SIZE)
            real = data[idx]
            noise = np.random.normal(0,1,(BATCH_SIZE,LATENT_DIM))

            with tf.GradientTape() as tape:
                fake = gen(noise, training=True)
                real_out = critic(real, training=True)
                fake_out = critic(fake, training=True)

                gp = gradient_penalty(critic, real, fake)

                c_loss = tf.reduce_mean(fake_out) - tf.reduce_mean(real_out) + LAMBDA_GP * gp

            grads = tape.gradient(c_loss, critic.trainable_variables)
            c_opt.apply_gradients(zip(grads, critic.trainable_variables))

        # ── Train Generator ──
        noise = np.random.normal(0,1,(BATCH_SIZE,LATENT_DIM))

        with tf.GradientTape() as tape:
            fake = gen(noise, training=True)
            fake_out = critic(fake, training=True)
            g_loss = -tf.reduce_mean(fake_out)

        grads = tape.gradient(g_loss, gen.trainable_variables)
        g_opt.apply_gradients(zip(grads, gen.trainable_variables))

        # ── Logging ──
        if step % 50== 0:
            print(f"{step} | C loss: {c_loss:.3f} | G loss: {g_loss:.3f}")
            save_images(gen, step)

    gen.save(f"{SAVE_DIR}/generator.h5")

# ─── RUN ─────────────────────────────
if __name__ == "__main__":
    train()

0 | C loss: 7.572 | G loss: 0.444
50 | C loss: -12.227 | G loss: 23.336
100 | C loss: -7.753 | G loss: 1.585
150 | C loss: -4.593 | G loss: 2.836
200 | C loss: -5.642 | G loss: 3.953
250 | C loss: -4.820 | G loss: 0.308
300 | C loss: -4.220 | G loss: -2.668
350 | C loss: -2.697 | G loss: 5.777
400 | C loss: -2.555 | G loss: 0.950
450 | C loss: -2.779 | G loss: -9.035
500 | C loss: -7.254 | G loss: -21.161
550 | C loss: -2.209 | G loss: 13.297
600 | C loss: -2.984 | G loss: -7.439
650 | C loss: -2.464 | G loss: -11.220
700 | C loss: -3.475 | G loss: -6.197
750 | C loss: -4.255 | G loss: -29.403
800 | C loss: -3.378 | G loss: -20.087
850 | C loss: -3.810 | G loss: -34.018
900 | C loss: -2.802 | G loss: -68.501
950 | C loss: -4.579 | G loss: -65.505
1000 | C loss: -3.024 | G loss: -56.037
1050 | C loss: -2.368 | G loss: -48.849
1100 | C loss: -3.220 | G loss: -32.926
1150 | C loss: -5.856 | G loss: -58.337
1200 | C loss: -2.918 | G loss: -37.581
1250 | C loss: -2.835 | G loss: -26.851
130